# Week 60, Day 5: Functional style in Python

Companion notebook. The interactive figures, X-ray views, quizzes and auto-marked exercises are on the [lesson page](https://peter-adepoju.github.io/python-atlas/days/w060d5.html). Use this notebook to keep your own copy of the code.

**How to use it:** run each cell with Shift+Enter. Change the code and run it again: that's how you learn.

In [ ]:
# Run this cell once. It lets you check exercise answers inside this notebook.
# Workflow: run your exercise cell, then immediately run the "check" cell below it.
import io, builtins

def check(tests_src):
    """Run an exercise's tests against the cell you ran just before this one."""
    ip = get_ipython()
    code = ip.user_ns["In"][-2]
    def run(feed=(), echo=False):
        out = io.StringIO(); q = [str(x) for x in feed]; real_in, real_print = builtins.input, builtins.print
        def fake_input(p=""):
            if not q:
                raise AssertionError("The program asked for more input than the test gave it.")
            v = q.pop(0); out.write(str(p) + (v + "\n" if echo else "")); return v
        def fake_print(*a, **k):
            k.setdefault("file", out); real_print(*a, **k)
        builtins.input, builtins.print = fake_input, fake_print
        try:
            exec(code, {"__name__": "__main__"})
        finally:
            builtins.input, builtins.print = real_in, real_print
        return out.getvalue()
    norm = lambda t: "\n".join(l.rstrip() for l in str(t).rstrip().splitlines())
    def expect_output(want, inputs=(), echo=False):
        got = run(inputs, echo)
        assert norm(got) == norm(want), f"expected {norm(want)!r} but got {norm(got)!r}"
    import re
    uses = lambda *names: all(re.search(r"\b" + re.escape(n) + r"\b", code) for n in names)
    def run_with(subs, feed=(), echo=False):
        nonlocal code
        saved = code
        for old, new in subs.items():
            code = code.replace(old, new, 1)
        try:
            return run(feed, echo)
        finally:
            code = saved
    env = dict(ip.user_ns, run=run, norm=norm, source=code, expect_output=expect_output, uses=uses, run_with=run_with)
    try:
        env["output"] = run()
    except AssertionError:
        env["output"] = ""
    exec(tests_src, env)
    tests = [(k, v) for k, v in env.items() if k.startswith("test_") and callable(v)]
    ok = 0
    for name, fn in tests:
        label = (fn.__doc__ or name[5:].replace("_", " ")).strip()
        try:
            fn(); ok += 1; print("✓", label)
        except AssertionError as e:
            print("✗", label + ":", e or "not quite yet")
    print(f"{ok} of {len(tests)} checks passed.")


**Predict before you run:** what will this print?



In [ ]:
def add_bonus_in_place(scores):
    for i in range(len(scores)):
        scores[i] += 5
    return scores

def add_bonus(scores):
    return [s + 5 for s in scores]

marks = [60, 70]
print(add_bonus(marks), marks)
print(add_bonus_in_place(marks), marks)

In [ ]:
def apply_discount(prices, percent):
    """Pure: returns new prices, leaves the old ones alone."""
    return [round(p * (100 - percent) / 100, 2) for p in prices]

def report(prices):
    """The side effect lives at the edge."""
    for p in prices:
        print(f"£{p:.2f}")

original = [10.0, 4.5]
report(apply_discount(original, 20))
print(original)

In [ ]:
from dataclasses import dataclass, replace
from typing import NamedTuple

class Point(NamedTuple):
    x: int
    y: int

@dataclass(frozen=True)
class Order:
    item: str
    quantity: int
    tags: tuple = ()

p = Point(1, 2)
q = p._replace(y=5)
o = Order("tea", 2)
o2 = replace(o, quantity=3, tags=o.tags + ("gift",))
print(p, q, o, o2, {o: "first"}[o])

In [ ]:
from operator import itemgetter, attrgetter, methodcaller, mul
from functools import reduce

rows = [{"name": "jam", "price": 3}, {"name": "tea", "price": 2}, {"name": "milk", "price": 2}]
print(sorted(rows, key=itemgetter("price", "name")))
print(list(map(int, ["1", "2", "3"])), [s.strip() for s in [" a ", "b "]], list(map(methodcaller("strip", "*"), ["*x*", "y*"])))
print(reduce(mul, [2, 3, 4]), sorted(["Bo", "ada", "Cy"], key=methodcaller("lower")))

> **Key idea.** A pure function's result depends only on its arguments and it changes nothing else, so it's easy to test, reason about, cache and combine; push side effects to the edges. Prefer immutable values (tuples, frozensets, NamedTuple, frozen dataclasses) and update by building new ones with _replace or dataclasses.replace. map and filter are lazy; comprehensions are usually clearer, map with an existing function is neat, and operator's itemgetter, attrgetter and methodcaller replace small lambdas. Compose small functions, left to right with a pipe. Use the style for transformations and testable logic; use loops for long iterations (no tail-call optimisation), names over lambda chains, and objects for state that really changes.

### Exercise: Pure versions

Write pure versions (none may change its arguments): `with_bonus(scores, bonus)` (a new dict with every score increased), `add_item(basket, item)` (a new **tuple** with the item added), `without(items, unwanted)` (a new list without any of the unwanted values), and `renamed(record, old, new)` (a new dict with key `old` renamed to `new`, the order of the other keys kept and the renamed key moved to the end).

In [ ]:
def with_bonus(scores, bonus):
    ___

def add_item(basket, item):
    ___

def without(items, unwanted):
    ___

def renamed(record, old, new):
    ___

In [ ]:
# Check: run the exercise cell above first, then this one.
check("import copy\ndef test_pure():\n    \"no side effects\"\n    scores, basket, items, rec = {\"ada\": 60, \"bo\": 70}, (\"tea\",), [1, 2, 3, 2], {\"a\": 1, \"b\": 2, \"c\": 3}\n    before = copy.deepcopy((scores, basket, items, rec))\n    assert with_bonus(scores, 5) == {\"ada\": 65, \"bo\": 75} and add_item(basket, \"jam\") == (\"tea\", \"jam\")\n    assert without(items, {2}) == [1, 3] and list(renamed(rec, \"a\", \"z\").items()) == [(\"b\", 2), (\"c\", 3), (\"z\", 1)]\n    assert (scores, basket, items, rec) == before\n    assert with_bonus(scores, 5) is not scores and isinstance(add_item((), 1), tuple)")

### Exercise: Operator functions

Using `operator.itemgetter`, `attrgetter` and `methodcaller` (no lambdas): `by_price_then_name(rows)` (dicts sorted by `"price"`, then `"name"`), `owners(accounts)` (the `owner.name` of each object, in order), `clean(words)` (each string stripped of spaces, then lower-cased, as two `map` calls with a `methodcaller` each), and `second_items(pairs)` (the second element of each pair).

In [ ]:
from operator import itemgetter, attrgetter, methodcaller

def by_price_then_name(rows):
    ___

def owners(accounts):
    ___

def clean(words):
    ___

def second_items(pairs):
    ___

In [ ]:
# Check: run the exercise cell above first, then this one.
check("from types import SimpleNamespace as NS\ndef test_getters():\n    \"operator\"\n    rows = [{\"name\": \"jam\", \"price\": 3}, {\"name\": \"tea\", \"price\": 2}, {\"name\": \"milk\", \"price\": 2}]\n    assert [r[\"name\"] for r in by_price_then_name(rows)] == [\"milk\", \"tea\", \"jam\"]\n    accts = [NS(owner=NS(name=\"Ada\")), NS(owner=NS(name=\"Bo\"))]\n    assert owners(accts) == [\"Ada\", \"Bo\"] and clean([\"  Tea \", \"JAM\"]) == [\"tea\", \"jam\"] and second_items([(1, \"a\"), (2, \"b\")]) == [\"a\", \"b\"]\n    assert \"lambda\" not in source")

### Exercise: Left to right

Write `pipe(value, *funcs)`: apply the functions to the value in order, left to right (`pipe(x, f, g) == g(f(x))`), and `pipeline(*funcs)`: a function that does the same to whatever it's given. Then `word_count = pipeline(...)`: a pipeline built only from existing functions and methods (no lambdas, no def) that takes a string and returns the number of words in it.

In [ ]:
from functools import reduce

def pipe(value, *funcs):
    ___

def pipeline(*funcs):
    ___

word_count = ___

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_pipes():\n    \"composition\"\n    assert pipe(\"  Hello World \", str.strip, str.lower) == \"hello world\" and pipe(5) == 5\n    shout = pipeline(str.strip, str.upper)\n    assert shout(\" hi \") == \"HI\" and pipeline()(3) == 3\n    assert word_count(\"the cat  sat \") == 3 and word_count(\"\") == 0\n    assert \"lambda\" not in source.split(\"word_count\")[1]")

### Exercise: Immutable records

Write a frozen dataclass `Item(name, price, quantity=1)` and a `NamedTuple` `Basket` with fields `items` (a tuple of `Item`) and `discount` (a percentage, default 0). Then pure functions: `add(basket, item)` (a new basket with the item, merging quantities if an item with the same name is already there), `set_discount(basket, percent)`, and `total(basket)` (the sum of price × quantity, less the discount, rounded to 2 places).

In [ ]:
from dataclasses import dataclass, replace
from typing import NamedTuple

___

In [ ]:
# Check: run the exercise cell above first, then this one.
check("import dataclasses\ndef test_frozen():\n    \"immutable updates\"\n    b0 = Basket(())\n    b1 = add(b0, Item(\"tea\", 2.5))\n    b2 = add(b1, Item(\"tea\", 2.5, 2))\n    b3 = add(b2, Item(\"jam\", 3.0))\n    assert b0.items == () and b1.items == (Item(\"tea\", 2.5, 1),) and b2.items == (Item(\"tea\", 2.5, 3),)\n    assert total(b3) == 10.5 and total(set_discount(b3, 10)) == 9.45 and b3.discount == 0\n    try:\n        b3.items[0].price = 1\n        assert False\n    except dataclasses.FrozenInstanceError:\n        pass\n    assert {Item(\"a\", 1): 1}[Item(\"a\", 1)] == 1")

### Exercise: Recursion or a loop?

Python has no tail-call optimisation, so a recursive "loop" over a long list hits the recursion limit. Write `sum_recursive(nums)` recursively (fine for short lists), then `sum_folded(nums)` with `functools.reduce` and `sum_looped(nums)` with a `for` loop, both working on a list of 100,000 numbers. Finally `deepest(n)`: return True if `sum_recursive(list(range(n)))` succeeds and False if it raises `RecursionError`.

In [ ]:
from functools import reduce

def sum_recursive(nums):
    ___

def sum_folded(nums):
    ___

def sum_looped(nums):
    ___

def deepest(n):
    ___

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_loops():\n    \"no tail calls\"\n    assert sum_recursive([1, 2, 3]) == 6 and sum_recursive([]) == 0\n    big = list(range(100_000))\n    assert sum_folded(big) == sum_looped(big) == sum(big)\n    assert deepest(100) is True and deepest(100_000) is False")

### Bug hunt: Pure on the outside only

`normalised` looks pure but sorts its argument **in place** before copying it, so the caller's list is reordered and this prints `[1, 2, 3] [1, 2, 3]`. Sort a copy so the original keeps its order: `[1, 2, 3] [3, 1, 2]`.

In [ ]:
def normalised(nums):
    nums.sort()
    return list(nums)

data = [3, 1, 2]
print(normalised(data), data)

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_output():\n    \"no mutation\"\n    expect_output(\"[1, 2, 3] [3, 1, 2]\")")

### Bug hunt: sort returns None

`list.sort()` changes the list and returns `None`, so this "functional" one-liner prints `None`. Use `sorted`, which returns a new list, so it prints `['ada', 'bo', 'cy']`.

In [ ]:
names = ["cy", "ada", "bo"]
print(list(names).sort())

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_output():\n    \"sorted\"\n    expect_output(\"['ada', 'bo', 'cy']\")")

### Bug hunt: A map used up

`map` returns a lazy iterator, used up by the first `sum`, so the `max` afterwards fails with `ValueError` and this prints `15 empty`. Make a list of the results so it prints `15 6`.

In [ ]:
lengths = map(len, ["tea", "jam", "cheese", "ab"])
total = sum(lengths) + 1
try:
    print(total, max(lengths))
except ValueError:
    print(total, "empty")

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_output():\n    \"keep the values\"\n    expect_output(\"15 6\")")

### Puzzle: An immutable update

Arrange the lines that add a tag to a frozen dataclass record, returning a new record. One line is a trap.

(Put these lines in order on the website.)

### Exercise: Predict them



In [ ]:
p = ___   # True or False: print is a pure function
q = ___   # (1, 2) + (3,)
r = ___   # sorted([{"n": 2}, {"n": 1}], key=itemgetter("n"))[0]["n"]
s = ___   # pipe(" A b ", str.strip, str.lower, str.split), with today's pipe
t = ___   # the exception when a recursive function goes about 1000 calls deep, as a string

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_values():\n    \"each\"\n    assert p is False and q == (1, 2, 3) and r == 1, \"p, q and r\"\n    assert s == [\"a\", \"b\"] and t == \"RecursionError\", \"s and t\"")

### Quiz

Take this quiz on the website: it is marked for you and feeds your daily review.

### Exercise: Challenge: a ledger of events

Instead of changing an account balance directly, record **events** and compute the state from them, purely. The state is a frozen dataclass `Account(balance=0, frozen=False, history=())`. Events are tuples: `("deposit", amount)`, `("withdraw", amount)`, `("freeze",)` and `("unfreeze",)`.

- `apply(state, event)`: return the **new** state. Deposits add; withdrawals subtract but are refused (the state is returned unchanged) if they'd take the balance below zero or the account is frozen; deposits to a frozen account are also refused. Every accepted event is appended to `history`. An unknown event raises `ValueError`.
- `replay(events, start=Account())`: the final state after every event, with `functools.reduce`.
- `states(events, start=Account())`: the list of every state, starting with `start`, with `itertools.accumulate`.
- `undo(events, n)`: the state after all but the last n events, by replaying (no mutation anywhere).

In [ ]:
from dataclasses import dataclass, replace
from functools import reduce
from itertools import accumulate

@dataclass(frozen=True)
class Account:
    balance: int = 0
    frozen: bool = False
    history: tuple = ()

def apply(state, event):
    ___

def replay(events, start=Account()):
    ___

def states(events, start=Account()):
    ___

def undo(events, n):
    ___

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_ledger():\n    \"pure state from events\"\n    events = [(\"deposit\", 100), (\"withdraw\", 30), (\"freeze\",), (\"withdraw\", 10), (\"unfreeze\",), (\"withdraw\", 100), (\"withdraw\", 20)]\n    final = replay(events)\n    assert final.balance == 50 and not final.frozen\n    assert final.history == ((\"deposit\", 100), (\"withdraw\", 30), (\"freeze\",), (\"unfreeze\",), (\"withdraw\", 20))\n    s = states(events)\n    assert len(s) == 8 and s[0] == Account() and s[3].frozen and s[4] == s[3]\n    assert undo(events, 1).balance == 70 and undo(events, 0) == final and undo(events, len(events)) == Account()\n    a = Account()\n    b = apply(a, (\"deposit\", 5))\n    assert a.balance == 0 and b.balance == 5\n    try:\n        apply(a, (\"explode\",))\n        assert False\n    except ValueError:\n        pass")

## Warm-up

Both return the same list of numbers, but only one leaves the caller's data alone. That difference is today's subject.

## Pure functions

A function is **pure** when its result depends only on its arguments, and calling it has no **side effects**: it doesn't change its arguments, global variables, files, the screen or the clock. `len`, `sorted`, `abs` and most maths are pure; `list.sort`, `print`, `random.random` and anything reading `input` aren't.

Pure functions are easy to:

- **test**: same input, same output, no setup or clean-up (Week 24's tests are simplest for pure code);
- **reason about**: you can read a call in isolation, and swap a call for its result;
- **cache**: `@cache` is only safe on pure functions (yesterday);
- **combine**: the output of one is safely the input of another.

Programs need side effects (that's how they do anything useful), so the functional habit is to **push them to the edges**: a thin layer reads input and writes output, and the logic in the middle is pure functions. Week 36's layered architecture had the same shape.

## Values that don't change

Functional style prefers **immutable** values: instead of changing a value, make a new one with the change. Then nothing that holds the old value is surprised, values can be shared freely, and they can be dict keys and set members. Python offers:

- `tuple` and `frozenset` instead of `list` and `set`;
- `typing.NamedTuple` and `@dataclass(frozen=True)` (Week 32) for records, updated with `record._replace(...)` or `dataclasses.replace(record, ...)`;
- strings, numbers and `None`, which are always immutable.

Building new values has a cost (copying), which is usually small. For big structures updated often, a mutable structure inside a function that returns its result is a fine compromise: the function is still pure from the outside.

## map, filter and operator

`map(func, iterable)` applies a function to each item; `filter(pred, iterable)` keeps the items where the predicate is true. Both are lazy iterators (Monday). In Python, a comprehension or generator expression is usually clearer than `map` or `filter` with a `lambda`; `map` reads best when the function already exists: `map(str.strip, lines)`, `map(int, fields)`.

The `operator` module gives the operators and common lookups as functions, avoiding small lambdas:

- `operator.add`, `mul`, `neg`, ... for `reduce` and `map`;
- `itemgetter("name")` or `itemgetter(1, 0)`: get keys or positions (great as a `key=`);
- `attrgetter("owner.name")`: get (dotted) attributes;
- `methodcaller("strip", "*")`: call a method with arguments.

## Composing

Small pure functions combine into bigger ones. `compose(f, g)` (yesterday) applies right to left, as in maths; a `pipe` applies **left to right**, which reads in the order things happen: `pipe(text, str.strip, str.lower, str.split)`. Monday's generator pipelines are the same idea for streams.

## Where it fits Python

Functional style helps when:

- the logic is a **transformation** of data: filtering, mapping, grouping, folding;
- code must be **tested** heavily or run in parallel (no shared changing state);
- values are shared between parts of a program that shouldn't affect each other.

It hurts when pushed further than Python supports:

- **Deep recursion** instead of loops: Python has no tail-call optimisation and a recursion limit of about 1000, so a recursive "loop" over a long list fails. Use loops, or `reduce`.
- **Chains of lambdas** are harder to read than a named function or a comprehension.
- **Copying big structures** in a tight loop is slow; mutate a local copy and return it.
- **State that genuinely changes** (a game, a bank account, a GUI) is often clearest as an object with methods.

Python's own guidance, "The Zen of Python" (`import this`), says "readability counts": pick whichever style reads best for the job.

## Practice

## Quiz

## Daily challenge

## Before you go

Pick one function from an earlier project (the Week 25 `spend` tracker, or Week 38's `shelf`) that changes its arguments, and sketch a pure version that returns the new data instead. Tomorrow is the Forge: a plugin toolkit built from closures, decorators and `functools`.